In [3]:
#!pip install --upgrade pip setuptools wheel
#!pip install -r requirements.txt --verbose

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from wordcloud import WordCloud

# Lecture des données initiales
X_train_init = pd.read_csv("../data/X_train_update.csv")
y_train_init = pd.read_csv("../data/Y_train_CVw08PX.csv")
X_test_init = pd.read_csv("../data/X_test_update.csv")

# Affichage des informations sur les datasets
print(f"Info X_train_init : {X_train_init.info()}")
print(f"Info Y_train_init : {y_train_init.info()}")
print(f"Info X_test_init : {X_test_init.info()}")

# Affichage des tailles des datasets
print(f"Taille X_train_init : {X_train_init.shape}")
print(f"Taille Y_train_init : {y_train_init.shape}")
print(f"Taille X_test_init : {X_test_init.shape}")

# Affichage du nombre de classes dans la variable cible
print(y_train_init['prdtypecode'].nunique())  # 27 classes

# Merge données d'entrainement
full_data = pd.merge(X_train_init, y_train_init, left_index=True, right_index=True)

# Suppression de la colonne Unnamed: 0_y qui est une colonne d'index inutile
full_data = full_data.drop(['Unnamed: 0_y'], axis=1)

# Renomage de la colonne Unnamed: 0_x en id et mise en index de cette colonne
full_data.rename(columns={'Unnamed: 0_x': 'id'}, inplace=True)
full_data.set_index(['id'], inplace=True)

# X_test_init.rename(columns={'Unnamed: 0': 'id'}, inplace=True)
# X_test_init.set_index(['id'], inplace=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 84916 entries, 0 to 84915
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Unnamed: 0   84916 non-null  int64 
 1   designation  84916 non-null  object
 2   description  55116 non-null  object
 3   productid    84916 non-null  int64 
 4   imageid      84916 non-null  int64 
dtypes: int64(3), object(2)
memory usage: 3.2+ MB
Info X_train_init : None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 84916 entries, 0 to 84915
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   Unnamed: 0   84916 non-null  int64
 1   prdtypecode  84916 non-null  int64
dtypes: int64(2)
memory usage: 1.3 MB
Info Y_train_init : None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13812 entries, 0 to 13811
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Unna

In [3]:
# Informations sur les données du dataset d'entrainement
print("Types de données :")
print(full_data.dtypes)
print("\nValeurs manquantes :")
print(full_data.isnull().sum())
print("\nPourcentage de valeurs manquantes :")
print((full_data.isnull().sum() / len(full_data) * 100))

# nan_count = (full_data['description'].astype(str) == 'nan').sum()
# print(f"\nNombre de description NaN : {nan_count}")

Types de données :
designation    object
description    object
productid       int64
imageid         int64
prdtypecode     int64
dtype: object

Valeurs manquantes :
designation        0
description    29800
productid          0
imageid            0
prdtypecode        0
dtype: int64

Pourcentage de valeurs manquantes :
designation     0.000000
description    35.093504
productid       0.000000
imageid         0.000000
prdtypecode     0.000000
dtype: float64


In [4]:
# Statistiques descriptives pré-processing

print("\nStatistiques descriptives :")
# Longueur des textes
full_data['len_designation'] = (full_data['designation'].astype(str).apply(len))
full_data['len_description'] = (full_data['description'].astype(str).apply(len))

# Nombre de mots
full_data['nb_words_designation'] = (full_data['designation'].astype(str).apply(lambda x: len(x.split())))
full_data['nb_words_description'] = (full_data['description'].astype(str).apply(lambda x: len(x.split())))

# Affichage des statistiques descriptives pour les colonnes de texte
print("\nDESIGNATION")
print(f"Longueur minimale du titre : {full_data['len_designation'].min()} caractères")
print(f"Longueur maximale du titre : {full_data['len_designation'].max()} caractères")
print(f"Longueur moyenne du titre : {full_data['len_designation'].mean():.2f} caractères")
print(f"Longueur médiane du titre : {full_data['len_designation'].median()} caractères")
print(f"Nombre moyen de mots : {full_data['nb_words_designation'].mean():.2f} mots")


print("\nDESCRIPTION")
print(f"Longueur minimale de la description : {full_data['len_description'].min()} caractères")
print(f"Longueur maximale de la description : {full_data['len_description'].max()} caractères")
print(f"Longueur moyenne de la description : {full_data['len_description'].mean():.2f} caractères")
print(f"Longueur médiane de la description : {full_data['len_description'].median()} caractères")
print(f"Nombre moyen de mots : {full_data['nb_words_description'].mean():.2f} mots")




Statistiques descriptives :

DESIGNATION
Longueur minimale du titre : 11 caractères
Longueur maximale du titre : 250 caractères
Longueur moyenne du titre : 70.16 caractères
Longueur médiane du titre : 64.0 caractères
Nombre moyen de mots : 11.56 mots

DESCRIPTION
Longueur minimale de la description : 1 caractères
Longueur maximale de la description : 12451 caractères
Longueur moyenne de la description : 525.61 caractères
Longueur médiane de la description : 231.0 caractères
Nombre moyen de mots : 80.52 mots


In [5]:
# Comptage des catégories
count_cat = full_data['prdtypecode'].value_counts().sort_index()
print(f"Nombre de catégories : {len(count_cat)}")

# Top 10 des catégories les plus fréquentes
print("\nTop 10 des catégories les plus fréquentes :")
print(count_cat.sort_values(ascending=False).head(10))

Nombre de catégories : 27

Top 10 des catégories les plus fréquentes :
prdtypecode
2583    10209
1560     5073
1300     5045
2060     4993
2522     4989
1280     4870
2403     4774
2280     4760
1920     4303
1160     3953
Name: count, dtype: int64


In [12]:
#Fonction de nettoyage du texte simple pour les colonnes de texte
import re

def clean_text(text):
    if pd.isnull(text):
        return ""

    # Supprimer balises HTML
    text = re.sub(r'<.*?>', '', text)

    # Remplacer <br /> par un espace
    text = text.replace(r'<br />', ' ')

    # Remplacer les référence de caractère HTML
    text = text.replace(r'&amp;', '&')
    text = text.replace(r'&nbsp;', ' ')
    text = text.replace(r'&lt', '<')
    text = text.replace(r'&gt', '>')
    text = text.replace(r'&quot', '"')
    text = text.replace(r'&#39', "'")
    text = text.replace(r'&eacute', 'e')
    text = text.replace(r'&egrave', 'e')
    text = text.replace(r'&ecirc', 'e')
    
    # Convertir en minuscules
    text = text.lower()

    # Supprimer les espaces supplémentaires
    text = re.sub(r'\s+', ' ', text).strip()

    # Supprimer la ponctuation
    text = re.sub(r'[^\w\s]', '', text)

    return text

In [13]:
# Nettoyage du texte simple pour les colonnes de texte
full_data['clean_designation'] = full_data['designation'].apply(clean_text)
full_data['clean_description'] = full_data['description'].apply(clean_text)

# Vérification de doublon dans les désignations et descriptions nettoyées
duplicate_designation = full_data['clean_designation'].duplicated().sum()
duplicate_description = full_data['clean_description'].duplicated().sum()

# Concaténation des désignations et descriptions nettoyées pour l'analyse de texte
full_data['text'] = full_data['clean_designation'] + ' ' + full_data['clean_description']

# Affichage du nombre de doublons
print(f"Nombre de doublons dans les désignations nettoyées : {duplicate_designation}")
print(f"Nombre de doublons dans les descriptions nettoyées : {duplicate_description}")

Nombre de doublons dans les désignations nettoyées : 2689
Nombre de doublons dans les descriptions nettoyées : 37779


In [14]:
# Compte du nombre de descriptions NaN après nettoyage
nan_count = (full_data['clean_description'].astype(str) == 'nan').sum()
print(f"\nNombre de description NaN : {nan_count}")

# Création d'un nouveau dataframe "nettoyé" pour l'analyse de texte
clean_data = full_data.drop(columns=['designation', 'description', 'productid', 'imageid','clean_designation','clean_description', 'len_description', 'len_designation', 'nb_words_description', 'nb_words_designation'], axis=1)

# Création du dictionnaire des catégories
category_mapping = pd.read_csv("../data/dictionnaire.csv", encoding='utf-8')
dictionnaire = category_mapping.to_dict()['catégorie']


Nombre de description NaN : 0


# DataFrame "full_data" contenant les données d'entrainement avec les statistiques descriptives et les colonnes de texte avant nettoyage
# DataFrame "clean_data" contenant les données d'entrainement avec uniquement les colonnes de texte nettoyées et la variable cible

In [19]:
clean_data.head(20)

,prdtypecode,text
id,,
0,10,olivia personalisiertes notizbuch 150 seiten ...
1,2280,journal des arts le n 133 du 28092001 lart et...
2,50,grand stylet ergonomique bleu gamepad nintendo...
3,1280,peluche donald europe disneyland 2000 marion...
4,2705,la guerre des tuques luc a des idees de grande...
5,2280,afrique contemporaine n 212 hiver 2004 dossie...
6,10,christof e bildungsprozessen auf der spur
7,2522,conquérant sept cahier couverture polypro 240 ...
8,1280,puzzle scoobydoo avec poster 2x35 pieces


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier 
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.feature_extraction.text import TfidfVectorizer

# Importation de TfidfVectorizer pour la vectorisation du texte (Possibilité d'ajouter des stopwords dans TFIDF avec 'stop_words=fr_stop_words')
vectorizer = TfidfVectorizer()

# Création des ensembles d'entraînement et de test pour l'analyse de texte (20% pour le test)
X_train, X_test, y_train, y_test = train_test_split(test_data['text'], test_data['prdtypecode'], test_size=0.2, random_state=42)

# Vectorisation du texte
X_train = vectorizer.fit_transform(X_train)
X_test = vectorizer.transform(X_test)

In [ ]:
# Analyse des différentes langues du jeu de données
# from langdetect import detect, DetectorFactory, LangDetectException
# USE_LANG_DETECT = True  # Mettre à True pour activer la détection de langue
# DetectorFactory.seed = 0  # Pour des résultats reproductibles

# def detect_language(text):
#    if not USE_LANG_DETECT:
#        return 'unknown'
#    try:
#        return detect(text)
#    except (LangDetectException, Exception):
#        return 'unknown'
    

#test_data['language'] = test_data['text'].apply(detect_language)
#print(test_data['language'].value_counts())

In [17]:
# Analyse des différentes langues du jeu de données
import langdetect
from langdetect import detect_langs, DetectorFactory, LangDetectException

USE_LANG_DETECT = True  # Mettre à True pour activer la détection de langue
lang = detect_langs(str(clean_data['text'].head(1)))
print(f"Langues détectées dans clean_data: {lang}")

lang1 = detect_langs(str(X_train_init['designation'].head(1)))
print(f"Langues détectées dans X_train_init: {lang1}")

Langues détectées dans clean_data: [en:0.9999942938252357]
Langues détectées dans X_train_init: [en:0.714282951971165, fr:0.1428579275135321, de:0.14285790718999136]


In [ ]:
# Analyse et suppression des stopwords avec nltk
import nltk
from nltk.corpus import stopwords

# Création de stopwords pour le français, l'anglais et l'allemand
nltk.download('stopwords') # Téléchargement des stopwords si ce n'est pas déjà fait
fr_stop_words = set(stopwords.words('french'))
en_stop_words = set(stopwords.words('english'))
ger_stop_words = set(stopwords.words('german'))
# Création d'un ensemble de stopwords combiné pour le français, l'anglais et l'allemand
global_stop_words = fr_stop_words.union(en_stop_words).union(ger_stop_words)

USE_NLTK = True  # Mettre à True pour activer la suppression des stopwords avec nltk

# Fonction suppression des stopwords

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Yacinou\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
